# ICC Strategy — Meta-Labeling Model### End-to-end training notebook**Goal:** train a model that filters ICC signals — keeping trades that reach thetake-profit target, suppressing those that end at the stop.**Architecture — meta-labeling:**- **Primary model** = the ICC state machine. Decides *direction*. Rule-based, unchanged.- **Meta-model** = this notebook's classifier. Decides *take or skip*.Work through the sections in order; each depends on the previous one.| Section | Stage ||---|---|| 1 | Setup & configuration || 2 | Data loading & quality gate || 3 | Indicators (Layer 1) || 4 | Features (Layer 2) || 5 | ICC signals (primary model) || 6 | Meta-labels || 7 | Feature validation || 8 | Purged walk-forward training || 9 | Backtest & live parity || 10 | Export |Reference: `docs/01`–`docs/05`.

## 1. Setup & configuration

In [ ]:
import sys, warningsfrom pathlib import Pathfrom datetime import datetimeimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT / "src"))warnings.filterwarnings("ignore")pd.set_option("display.max_columns", 60)import icc_mlprint("icc_ml", icc_ml.__version__)

In [ ]:
from icc_ml.config import StrategyConfig, ExecutionConfig, SymbolSpec, SYMBOL_PRESETS# ---- EDIT THIS BLOCK with your broker's real values (see docs/05) ----SYMBOL    = "XAUUSD"TIMEFRAME = "H1"spec = SymbolSpec(    name=SYMBOL,    digits=2,                 # Symbol specification -> Digits    point=0.01,               # Symbol specification -> Point    contract_size=100,        # Symbol specification -> Contract size    typical_spread_points=25, # observed in Market Watch    commission_per_lot_roundturn=7.0,)cfg = StrategyConfig(    htf_pivot_len=2,    ltf_pivot_len=1,    tp_pips=2500,             # 2500 * 0.01 = $25.00 on XAUUSD    sl_swing_timeframe="4h",    sl_swing_pivot_len=2,    max_hold_bars=2000,)execc = ExecutionConfig(slippage_points=5.0, both_hit_same_bar_policy="sl_first")print(f"pip size      : {spec.pip}")print(f"TP distance   : {spec.pips_to_price(cfg.tp_pips)} price units")if spec.digits in (3, 5) and cfg.tp_pips > 600:    print("WARNING: on a 3/5-digit quote this TP is likely unreachable.")

## 2. Data loading & quality gateChoose **one** source. Option A is preferred: training on the same feed youtrade removes a whole class of discrepancies.

In [ ]:
from icc_ml.data_fetch import get_ohlcv, validate_ohlcv# ---- Option A: MetaTrader 5 (Windows, terminal running) ----# df, report = get_ohlcv(SYMBOL, TIMEFRAME, datetime(2016,1,1), datetime(2026,1,1), source="mt5")# ---- Option B: CSV export ----# df = pd.read_csv(ROOT / "data/raw/XAUUSD_H1.csv", parse_dates=["time"])# report = validate_ohlcv(df)# ---- Option C: synthetic (demo only — random walk, no real edge exists) ----from icc_ml.synthetic_data import generate_synthetic_ohlcvdf = generate_synthetic_ohlcv(n_bars=20000, start_price=2000.0, seed=7)report = validate_ohlcv(df)print(f"{len(df):,} bars   {df['time'].min()} -> {df['time'].max()}")report

In [ ]:
# Quality gate — stop here if anything failsproblems = []if report["duplicate_timestamps"]: problems.append("duplicate timestamps")if report["non_monotonic"]:        problems.append("non-monotonic time")if report["zero_or_negative_price_rows"]: problems.append("zero/negative prices")if report["high_less_than_low_rows"]:     problems.append("high < low")assert not problems, f"Fix these before continuing: {problems}"print("Data quality gate passed")df.set_index("time")["close"].plot(figsize=(14,4), title=f"{SYMBOL} {TIMEFRAME}")plt.show()

## 3. Indicators — Layer 1Raw indicator values across all 10 categories. Every function uses onlybackward-looking windows. The fractal swing detector is the one place needingcare (fractals are naturally centred/lookahead), so it tracks a `confirmed_at`index and never exposes a swing before it was knowable live.

In [ ]:
from icc_ml.indicators import compute_all_indicatorsind = compute_all_indicators(df)print(f"{ind.shape[1]} raw indicator columns")[c for c in ind.columns][:40]

## 4. Features — Layer 2Raw indicators become **Direction / Strength / Acceleration** (continuouscategories) and **State / Magnitude / Recency** (event categories)."Is the market trending?" is a weaker question than "is the trend strengtheningor decaying?" — the acceleration level is where much of the value sits.

In [ ]:
from icc_ml.features import build_all_featuresfeats = build_all_features(ind)feature_names = [c for c in feats.columns if c != "time"]print(f"{len(feature_names)} engineered features")groups = {    "Trend": ["ema","adx","di_","aroon","vortex","supertrend","linreg","macd"],    "Momentum": ["rsi","stoch","cci","roc","willr"],    "Volatility": ["atr","bb_","hv_","squeeze"],    "Structure": ["structure","bos","choch","swing"],    "Volume": ["obv","volume","vwap","mfi","cmf"],    "MeanRev": ["zscore","hurst","distance_pct","extreme"],    "Regime": ["regime"],    "PriceAction": ["candle","wick","body","consecutive","engulf","streak"],    "Liquidity": ["fvg","equal_","sweep","pdh","pdl"],    "Time": ["session","hour","dow"],}pd.DataFrame([    {"Category": g, "Features": sum(any(k in c for k in ks) for c in feature_names)}    for g, ks in groups.items()])

## 5. ICC signals — the primary modelThe state machine: **Indication** (HTF break) → **Correction** (LTF pivot arms atrigger) → **Continuation** (trigger crossover = entry).Signals fire on bar `t`; fills happen at `open[t+1]`. Filling at `close[t]` wouldbe lookahead.

In [ ]:
from icc_ml.strategy_icc import generate_icc_signalssignals = generate_icc_signals(df, cfg, spec)n_sig = int((signals["signal"] != 0).sum())print(f"signals : {n_sig}")print(f"long    : {int((signals['signal']==1).sum())}")print(f"short   : {int((signals['signal']==-1).sum())}")print(f"per 1k bars: {n_sig/len(df)*1000:.1f}")if n_sig < 300:    print("\nWARNING: <300 signals. Extend history before trusting walk-forward.")

## 6. Meta-labelsEach signal is simulated to its real exit using the strategy's own SL (H4 swing)and TP (`tp_pips`), with spread, slippage and commission applied.**`enforce_one_position=False` here is deliberate.** The model learns "is thissetup good?" — a property of market state, not of whether an unrelated earliertrade was open. Enforcing the block during labeling discards ~90% of trainingdata and makes the target depend on scheduling luck. The constraint is appliedlater, in the backtest, where it belongs. See `docs/03`.

In [ ]:
from icc_ml.icc_labeling import simulate_icc_trades, diagnose_trades, attach_features_to_tradestrades = simulate_icc_trades(df, signals, cfg, spec, execc, enforce_one_position=False)diag = diagnose_trades(trades, cfg, spec)for k in ["n_taken","win_rate","tp_hit_rate","sl_hit_rate","timeout_rate","avg_net_pips"]:    v = diag[k]    print(f"{k:16s}: {v:.4f}" if isinstance(v,float) else f"{k:16s}: {v}")for w in diag.get("warnings", []):    print(f"\nWARNING: {w}")

In [ ]:
# Baseline to beat. A model at 55% accuracy adds nothing if take-all was already 55%.BASELINE_WIN_RATE = diag["win_rate"]print(f"Baseline win rate (take every signal): {BASELINE_WIN_RATE:.1%}")trades["exit_reason"].value_counts().plot(kind="bar", figsize=(7,3), title="Exit reasons")plt.show()

In [ ]:
from icc_ml.train import get_feature_columnsjoined = attach_features_to_trades(trades, feats).dropna(subset=["win"])all_cols = get_feature_columns(joined)feature_cols = [c for c in all_cols                if joined[c].notna().mean() > 0.9 and joined[c].nunique(dropna=True) > 1]joined = joined.dropna(subset=feature_cols)print(f"training matrix: {len(joined)} trades x {len(feature_cols)} features")print(f"dropped {len(all_cols)-len(feature_cols)} constant/sparse features")

## 7. Feature validation — *before* trainingInformation Coefficient = Spearman rank correlation with the label.- |IC| of 0.05–0.35 is realistic for tradable features- |IC| > 0.9 means **leakage** — investigate immediately

In [ ]:
from icc_ml.validation import compute_ic, redundancy_clustersX, y = joined[feature_cols], joined["win"]ic = compute_ic(X, y).dropna()print(f"max |IC| = {ic.abs().max():.3f}")if ic.abs().max() > 0.9:    print("LEAKAGE SUSPECTED — stop and investigate.")ic.head(20).plot(kind="barh", figsize=(8,6), title="Top 20 features by |IC|")plt.gca().invert_yaxis(); plt.show()

In [ ]:
clusters = redundancy_clusters(X, corr_threshold=0.9)print(f"{len(clusters)} redundant clusters (keep the highest-IC member of each)\n")for c in clusters[:10]:    best = ic.reindex(c).abs().idxmax()    print(f"  keep {best:35s} <- {c}")

## 8. Purged walk-forward trainingThree things make this trustworthy:1. **Purging** — ICC trades hold hundreds of bars, so any training trade   resolving after the test window starts is dropped. Expect to lose a   meaningful share of rows; that cost is real, not a bug.2. **Threshold on expected pips**, not accuracy — payoffs are asymmetric.   Chosen on the *training* fold only.3. **Honest baseline** — every fold reports take-all performance.

In [ ]:
from icc_ml.train import train_walk_forward, summarize_foldsfold_results, oos = train_walk_forward(    joined, feature_cols, n_folds=5, embargo_bars=50, calibrate=True)summary = summarize_folds(fold_results)print(f"folds            : {summary['n_folds']}")print(f"mean AUC         : {summary['mean_auc']:.3f}")print(f"Brier score      : {summary['mean_brier']:.3f}")print(f"model pips       : {summary['total_model_pips']:+,.0f}")print(f"baseline pips    : {summary['total_baseline_pips']:+,.0f}")print(f"EDGE             : {summary['total_edge_pips']:+,.0f}")print(f"folds +ve edge   : {summary['folds_with_positive_edge']}/{summary['n_folds']}")print(f"trade reduction  : {summary['trade_reduction']:.1%}")

In [ ]:
pd.DataFrame(summary["per_fold"])[    ["fold","n_train","n_purged","auc","threshold",     "model_trades","model_net_pips","baseline_trades","baseline_net_pips"]]

In [ ]:
# Verdictedge_ok  = summary["total_edge_pips"] > 0folds_ok = summary["folds_with_positive_edge"] >= max(1, int(0.8*summary["n_folds"]))if edge_ok and folds_ok:    print("PASS — consistent positive edge.")elif edge_ok:    print("MARGINAL — positive total edge but inconsistent across folds.")else:    print("NO EDGE over baseline. Do not deploy.")    print("On synthetic random-walk data this is the CORRECT result.")

## 9. Backtest & live parityThe one-position constraint is applied **here**, sequentially — this is what asingle account could actually have captured.

In [ ]:
from icc_ml.backtest import compare_model_vs_baseline, sequential_backtest, live_parity_checkscmp_ = compare_model_vs_baseline(oos, spec, lots=cfg.fixed_lots)executed = sequential_backtest(oos, use_model_filter=True)base_exec = sequential_backtest(oos, use_model_filter=False)pd.DataFrame([    {"Strategy":"Model", **{k:v for k,v in cmp_["model"].items() if k!="error"}},    {"Strategy":"Take all", **{k:v for k,v in cmp_["baseline_take_all"].items() if k!="error"}},]).set_index("Strategy").T

In [ ]:
if not executed.empty:    fig, ax = plt.subplots(figsize=(13,4))    ax.plot(executed["equity_pips"].values, label="Model-filtered")    if not base_exec.empty:        ax.plot(base_exec["equity_pips"].values, label="Take all", alpha=0.6)    ax.set_title("Equity curve (pips)"); ax.set_xlabel("Trade #")    ax.legend(); ax.grid(alpha=0.3); plt.show()

In [ ]:
parity = live_parity_checks(oos, executed, trades)for k, v in parity.items():    if k == "failed_checks": continue    mark = {True:"PASS", False:"FAIL"}.get(v, "info")    print(f"  [{mark}] {k}: {v}")

## 10. ExportRefit on all data and persist. The bundle stores the **feature column order**alongside the model — a silent column reorder at inference time is a classic wayto ship a broken live system.

In [ ]:
from icc_ml.train import fit_final_modelthreshold = float(np.mean([r.threshold for r in fold_results]))out_path = ROOT / "models" / f"model_icc_{SYMBOL}_{TIMEFRAME}.joblib"out_path.parent.mkdir(exist_ok=True)bundle = fit_final_model(joined, feature_cols, threshold, out_path=str(out_path))print(f"saved {out_path}")print(f"  trades   : {bundle['n_training_trades']}")print(f"  features : {len(bundle['feature_cols'])}")print(f"  threshold: {bundle['threshold']:.3f}")

In [ ]:
# Deployment gate — all must be True (see docs/04 §7)gate = {    "TP reachable (>5%)":        diag["tp_hit_rate"] > 0.05,    ">=300 labeled trades":      len(joined) >= 300,    "Edge in >=80% of folds":    summary["folds_with_positive_edge"] >= max(1,int(0.8*summary["n_folds"])),    "Beats take-all baseline":   summary["total_edge_pips"] > 0,    "Timeout rate <30%":         diag["timeout_rate"] < 0.30,    "Parity checks passed":      parity["all_passed"],}for k,v in gate.items():    print(f"  [{'PASS' if v else 'FAIL'}] {k}")print("\nREADY — forward-test on demo >=1 month before live."      if all(gate.values()) else      "\nNOT READY. A strategy with no edge is not fixed by a better model.")